# 0. The final model and its settings

Two files define the model (plan claude/final_model_plan_2026-11-03.md, FM-01, FM-02):
`config/model.par` holds every openCARP setting; `config/model.json` holds the sweep variables
(baseline and planned values) and the settings openCARP cannot read. Nothing is simulated here.

| Notebook | Stages | What it does |
| --- | --- | --- |
| 01 | preflight, conditioning, tune, simulation | beta tuning and the openCARP runs (hours on the workstation) |
| 02 | physiology, window | activation checks; measurement window |
| 03 | electrograms | unipolar phi_e at every electrode (node-set electrodes) |
| 04 | measurements, export, changePoints | acquisition chain, PP, peak frequency; Wang's code |
| 05 | groundTruth | contributing tissue: classes, depth profile, 3D fields |
| 06 | report | acceptance, report.md, figures |
| 07 | sweeps | the one-at-a-time sweep |

One command runs the same stages: `python -m efov run --out runs/v4_reference --np 24`.

In [ ]:
import os, sys
from pathlib import Path
REPOSITORY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPOSITORY))

# ---- settings: edit here, or set EFOV_CONFIG / EFOV_OUT / EFOV_NP ----
CONFIG = os.environ.get("EFOV_CONFIG", str(REPOSITORY / "config" / "model.json"))
OUT    = os.environ.get("EFOV_OUT",    str(REPOSITORY / "runs" / "v4_reference"))
NP     = int(os.environ.get("EFOV_NP", "24"))      # MPI processes for openCARP
# openCARP's own MPI launcher (efov-runner image); needed when NP > 1
os.environ.setdefault("MPIEXEC_BIN", "/usr/local/lib/opencarp/lib/petsc/bin/mpiexec")

import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from efov.config import loadConfig
from efov.pipeline import Run

config = loadConfig(CONFIG)
run = Run(config, OUT, processCount=NP)
print(f"{config['modelName']} -> {run.out}")
run.status()

## The openCARP parameter file (every simulator setting)

In [ ]:
print(open(config["modelPar"]).read())

## Sweep variables: baseline and planned values

In [ ]:
pd.DataFrame({k: dict(baseline=v["baseline"], planned=v["planned"], actsOn=v.get("actsOn", ""))
              for k, v in config["sweep"].items()}).T

## Size of the run and assumptions in force

In [ ]:
import subprocess
print(subprocess.run([sys.executable, "-m", "efov", "plan", "--config", CONFIG], cwd=REPOSITORY, capture_output=True, text=True).stdout)